# FabriX-QA — YOLOv8n baseline v1 (2026-09-10)

Fine-tune **COCO-pretrained `yolov8n.pt`**, not a randomly initialized model, on the processed **semantic detector** dataset. Baseline v1 uses all semantic train images, including disclosed synthetic examples and weak full-image rmshashi labels. It is **not** the strong-label-only baseline or proof that synthetic augmentation helps.

1. Upload this notebook to Google Colab. Select **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU), then run cells in order.
2. Upload your trusted `processed.zip` to Drive and edit `DATASET_ZIP` in cell 4. Include the semantic `data.yaml`, `images/{train,val,test}`, `labels/{train,val,test}`, and `manifest.jsonl`. Both a wrapping `processed/` directory and a flat archive work. The full processed archive also works; anomaly files are never selected for this detector.
3. Authorize Drive mounting. Installation and pretrained-weight download require internet. Keep sufficient Colab disk space for extraction and Drive space for checkpoints.
4. Inspect the printed YAML and actual counts before running training. The September 5 build has **4,607/210/208 detector images**, including **390 synthetic train images**. There are **nine stable IDs: seven populated defect classes plus two reserved unmapped IDs**; normal/background images have empty labels. Do not renumber labels to match an assumed eight-class count.
5. Train for up to 80 epochs with validation-based early stopping. For GPU out-of-memory errors, lower `BATCH_SIZE` from 8 to 4 and start a new run. Run folders and checkpoints are saved to Drive throughout training; cell 8 also saves the requested top-level `best.pt`.
6. Share the validation report, counts, run metadata and training curves with the team. Test labels are counted for integrity only: **no test inference or tuning** occurs here.

This notebook has local structural/helper checks only. **Colab installation, GPU training and measured metrics still require Zarwan's manual execution.** Nothing below contains precomputed results.

API references: [Ultralytics training](https://docs.ultralytics.com/modes/train/), [validation](https://docs.ultralytics.com/modes/val/), [pinned package](https://pypi.org/project/ultralytics/8.4.133/).

In [ ]:
# Mount your own Google Drive; approve access in Colab.
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# Run in a fresh Colab GPU runtime. Do not install the local CPU-oriented
# ai/requirements.txt here: retain Colab's compatible CUDA torch/torchvision.
import subprocess
import sys
from importlib import metadata

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "ultralytics==8.4.133",
        "PyYAML==6.0.3",
    ]
)
import torch
import ultralytics

print("Python:", sys.version)
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("torchvision:", metadata.version("torchvision"))
if not torch.cuda.is_available():
    raise RuntimeError(
        "Select a GPU runtime in Colab, reconnect, and rerun from cell 2."
    )
print("GPU:", torch.cuda.get_device_name(0))
# If Colab requests a runtime restart after pip, restart and rerun from cell 2.


In [ ]:
# EDIT THIS PATH to the actual location of your trusted processed.zip in Drive.
DATASET_ZIP = "/content/drive/MyDrive/FabriX-QA/processed.zip"

import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath


def extract_dataset(archive_path: Path, destination: Path) -> None:
    """Extract a trusted ZIP safely, without merging it into an older dataset."""
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Upload processed.zip or correct DATASET_ZIP: {archive_path}"
        )
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(
            f"{destination} is not empty. For the SAME completed extraction, skip "
            "this cell and run cell 5. For a different/partial archive, use a fresh "
            "runtime or change DATASET_DIR; this notebook never deletes your files."
        )
    with zipfile.ZipFile(archive_path) as archive:
        members = []
        targets = set()
        for info in archive.infolist():
            name = PurePosixPath(info.filename.replace("\\", "/"))
            if (
                name.is_absolute()
                or ".." in name.parts
                or any(":" in p for p in name.parts)
            ):
                raise ValueError(f"Unsafe ZIP member: {info.filename}")
            if stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError(f"ZIP symlinks are not accepted: {info.filename}")
            # Exclude recoverable preprocessing archives, previews and stale
            # Windows loader caches; none belong in an active Colab training set.
            if (
                any(p.startswith(("_", ".")) for p in name.parts)
                or name.suffix == ".cache"
            ):
                continue
            if info.is_dir():
                continue
            if name.as_posix() in targets:
                raise ValueError(f"Duplicate ZIP member: {name}")
            targets.add(name.as_posix())
            members.append((info, name))
        if not members:
            raise ValueError("Archive contains no usable dataset files.")
        required = sum(info.file_size for info, _ in members)
        destination.parent.mkdir(parents=True, exist_ok=True)
        free = shutil.disk_usage(destination.parent).free
        if required + 2 * 1024**3 > free:
            raise RuntimeError(
                f"Insufficient disk: need {required / 1024**3:.1f} GiB + 2 GiB reserve."
            )
        destination.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {len(members):,} files ({required / 1024**3:.2f} GiB)…")
        for info, name in members:
            target = destination.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open("xb") as sink:
                shutil.copyfileobj(source, sink)
    print(f"Dataset extracted to {destination}")


DATASET_DIR = Path("/content/dataset")
extract_dataset(Path(DATASET_ZIP), DATASET_DIR)


In [ ]:
# Count actual images/labels, not directory names or documented estimates.
import hashlib
import json
import math
from collections import Counter
from pathlib import Path

import yaml


def sha256_file(path: Path) -> str:
    """Stream hashes so large manifests/weights do not require duplicate RAM."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def inspect_dataset(dataset_dir: Path, config_dir: Path) -> tuple:
    """Select semantic YAML, validate labels/provenance, and write a Colab copy."""
    expected_names = [
        "hole",
        "weave_error",
        "stain",
        "foreign_object",
        "crease",
        "edge_damage",
        "pattern_break",
        "aitex_unmapped",
        "tilda_unmapped",
    ]
    matches = []
    for path in dataset_dir.rglob("data.yaml"):
        if any(p.startswith(("_", ".")) for p in path.relative_to(dataset_dir).parts):
            continue
        data = yaml.safe_load(path.read_text(encoding="utf-8"))
        if not isinstance(data, dict):
            continue
        names = data.get("names")
        if isinstance(names, dict) and set(names) == set(range(len(expected_names))):
            names = [names[i] for i in range(len(names))]
        if names == expected_names:
            matches.append((path, data))
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one semantic data.yaml, found {[str(p) for p, _ in matches]}"
        )
    original_yaml, original = matches[0]
    root = original_yaml.parent
    print("Original semantic YAML:", original_yaml)
    print(original_yaml.read_text(encoding="utf-8"))
    if original.get("nc", len(expected_names)) != len(expected_names):
        raise ValueError("nc and names disagree; do not silently remap class IDs.")
    manifest_path = root / "manifest.jsonl"
    if not manifest_path.is_file():
        raise FileNotFoundError(
            "Include manifest.jsonl to verify synthetic and split provenance."
        )
    records = {}
    lineage = {"parent_id": {}, "pixel_sha256": {}}
    with manifest_path.open(encoding="utf-8") as stream:
        for line in stream:
            row = json.loads(line)
            for key, split_by_id in lineage.items():
                identity = row[key]
                previous = split_by_id.setdefault(identity, row["split"])
                if previous != row["split"]:
                    raise ValueError(f"Cross-split {key} leakage: {identity}")
            if row.get("synthetic", False) and row["split"] != "train":
                raise ValueError("Synthetic manifest row found outside training.")
            if (
                row["track"] == "multiclass"
                and row["annotation_kind"] != "classification_only"
            ):
                if row["image"] in records:
                    raise ValueError(f"Duplicate manifest image: {row['image']}")
                records[row["image"]] = row
    suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    summary = {"splits": {}, "image_counts": {}, "synthetic_image_counts": {}}
    actual_keys = set()
    for split in ("train", "val", "test"):
        # Ignore stale absolute Windows path in the original YAML, but enforce
        # the known relative semantic layout rather than selecting anomaly/data.yaml.
        if original.get(split) not in (f"images/{split}", f"./images/{split}"):
            raise ValueError(f"Unexpected {split} layout: {original.get(split)}")
        image_dir, label_dir = root / "images" / split, root / "labels" / split
        images = sorted(p for p in image_dir.rglob("*") if p.suffix.lower() in suffixes)
        if not images or not label_dir.is_dir():
            raise ValueError(f"Missing/empty {split} images or labels.")
        counts, synthetic_counts = Counter(), Counter()
        expected_labels = set()
        synthetic_total = 0
        for image in images:
            key = image.relative_to(root).as_posix()
            actual_keys.add(key)
            row = records.get(key)
            label = label_dir / image.relative_to(image_dir).with_suffix(".txt")
            if label in expected_labels:
                raise ValueError(f"Two images share a label stem: {label}")
            expected_labels.add(label)
            if not label.is_file():
                raise FileNotFoundError(
                    f"Missing label (normals need an empty file): {label}"
                )
            if (
                row is None
                or row["split"] != split
                or row["label"] != label.relative_to(root).as_posix()
            ):
                raise ValueError(f"Manifest/file mismatch: {key}")
            synthetic = bool(row.get("synthetic", False))
            if synthetic != image.name.startswith("synthetic_"):
                raise ValueError(f"Synthetic filename/provenance mismatch: {key}")
            if split != "train" and (synthetic or row["augmented"]):
                raise ValueError(f"Held-out augmentation: {key}")
            classes = []
            for line in label.read_text(encoding="utf-8").splitlines():
                fields = line.split()
                if len(fields) != 5:
                    raise ValueError(f"Expected YOLO detect class x y w h: {label}")
                cls, x, y, w, h = map(float, fields)
                if not all(math.isfinite(v) for v in (cls, x, y, w, h)):
                    raise ValueError(f"Nonfinite label: {label}")
                if not cls.is_integer() or not 0 <= cls < len(expected_names):
                    raise ValueError(f"Invalid class ID: {label}")
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    raise ValueError(f"Invalid normalized coordinates: {label}")
                if (
                    min(x - w / 2, y - h / 2) < -1e-6
                    or max(x + w / 2, y + h / 2) > 1 + 1e-6
                ):
                    raise ValueError(f"Box exceeds image: {label}")
                classes.append(int(cls))
            if classes != row["classes"] or len(classes) != row["box_count"]:
                raise ValueError(
                    f"Manifest/label classes or box count disagree: {label}"
                )
            names = {expected_names[c] for c in classes} or {"normal/background"}
            counts.update(names)  # Count each image once per class, NOT each box.
            if synthetic:
                synthetic_total += 1
                synthetic_counts.update(names)
        if set(label_dir.rglob("*.txt")) != expected_labels:
            raise ValueError(f"Orphan labels in {split}.")
        summary["splits"][split] = {"images": len(images), "synthetic": synthetic_total}
        summary["image_counts"][split] = dict(counts)
        summary["synthetic_image_counts"][split] = dict(synthetic_counts)
    if actual_keys != set(records):
        raise ValueError("Semantic manifest and actual image inventory differ.")
    print(
        f"{'Class (images, not boxes)':<29} {'train':>7} {'val':>7} {'test':>7} {'synthetic train':>16}"
    )
    for name in expected_names + ["normal/background"]:
        totals = [
            summary["image_counts"][s].get(name, 0) for s in ("train", "val", "test")
        ]
        print(
            f"{name:<29} {totals[0]:7d} {totals[1]:7d} {totals[2]:7d} {summary['synthetic_image_counts']['train'].get(name, 0):16d}"
        )
    print("Actual split totals:", summary["splits"])
    print(
        "Counts can overlap for multi-class images; tiles/copies are not independent sources."
    )
    summary["manifest_sha256"] = sha256_file(manifest_path)
    summary["original_yaml_sha256"] = sha256_file(original_yaml)
    summary["names"] = expected_names
    print("Manifest SHA-256:", summary["manifest_sha256"])
    # Write ONLY a runtime config, preserving the original dataset YAML/labels.
    config_dir.mkdir(parents=True, exist_ok=True)
    runtime_yaml = config_dir / "data_colab.yaml"
    runtime_yaml.write_text(
        yaml.safe_dump(
            {
                "path": str(root.resolve()),
                "train": "images/train",
                "val": "images/val",
                "test": "images/test",
                "names": dict(enumerate(expected_names)),
                "nc": len(expected_names),
            },
            sort_keys=False,
        ),
        encoding="utf-8",
    )
    print("Colab runtime YAML:", runtime_yaml)
    print(runtime_yaml.read_text(encoding="utf-8"))
    return runtime_yaml, expected_names, summary


DATA_YAML, NAMES, DATASET_SUMMARY = inspect_dataset(
    DATASET_DIR, Path("/content/fabrix_config")
)


In [ ]:
# Baseline v1: all semantic training images (real + synthetic + weak labels).
# Modest photometric/flip augmentation; avoid mosaic/crops that erase tiny/edge defects.
# Edit BATCH_SIZE to 4 if a T4 reports CUDA out of memory. Never fall back silently to CPU.
BATCH_SIZE = 8
EPOCHS = 80
PATIENCE = 15
SEED = 42
DRIVE_ROOT = Path("/content/drive/MyDrive/FabriX-QA")

import uuid
from datetime import datetime, timezone

from ultralytics import YOLO, settings

if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable. Select a CUDA GPU runtime before training.")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive is not mounted; rerun cell 2.")
settings.update({"wandb": False, "mlflow": False, "comet": False})
run_name = (
    "baseline_v1_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid.uuid4().hex[:8]
)
RUN_DIR = DRIVE_ROOT / "runs" / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)
TRAIN_CONFIG = dict(  # noqa: C408 -- readable editable training keyword options
    data=str(DATA_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,
    batch=BATCH_SIZE,
    imgsz=640,
    device=0,
    workers=2,
    cache=False,
    seed=SEED,
    deterministic=True,
    pretrained=True,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_bias_lr=0.0,
    cos_lr=True,
    amp=True,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.0,
    hsv_s=0.1,
    hsv_v=0.1,
    val=True,
    plots=True,
    save=True,
    save_period=10,
    project=str(RUN_DIR.parent),
    name=RUN_DIR.name,
    exist_ok=True,
)
# A .pt checkpoint loads pretrained COCO weights; .yaml would start from scratch.
model = YOLO("yolov8n.pt")
pretrained_path = Path(model.ckpt_path)
RUN_METADATA = {
    "baseline": "v1",
    "notebook_dataset_as_of": "2026-09-10",
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_zip": DATASET_ZIP,
    "dataset": DATASET_SUMMARY,
    "training_config": TRAIN_CONFIG,
    "pretrained": "yolov8n.pt",
    "pretrained_sha256": sha256_file(pretrained_path),
    "python": sys.version,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_gib": torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "ultralytics": ultralytics.__version__,
    "evaluation_split": "val; final test remains unused",
    "training_scope": "all semantic real + synthetic + weak full-image labels",
}
(RUN_DIR / "run_metadata.json").write_text(
    json.dumps(RUN_METADATA, indent=2), encoding="utf-8"
)
(RUN_DIR / "pip-freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True),
    encoding="utf-8",
)
shutil.copy2(DATA_YAML, RUN_DIR / "data_colab.yaml")
print("Persistent Drive run directory:", RUN_DIR)
print("Checkpoints/logs save here during training; keep Drive mounted.")
train_results = model.train(**TRAIN_CONFIG)
BEST_WEIGHTS = Path(model.trainer.best)
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f"Training did not produce best.pt: {BEST_WEIGHTS}")
print("Training finished. Best checkpoint already on Drive:", BEST_WEIGHTS)


In [ ]:
# Re-evaluate the BEST checkpoint on validation only, never the final test set.
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=BATCH_SIZE,
    device=0,
    workers=2,
    conf=0.001,
    iou=0.7,
    augment=False,
    plots=True,
    project=str(RUN_DIR),
    name="best_validation",
    exist_ok=True,
)
VALIDATION_REPORT = {
    "split": "val",
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
    "per_class": {},
}
print(f"Validation mAP@0.5:      {metrics.box.map50:.4f}")
print(f"Validation mAP@0.5:0.95: {metrics.box.map:.4f}")
# ap_class_index maps metric-array positions to dataset IDs; absent classes
# must be reported as N/A, not assigned another class's score or the mean AP.
metric_indices = {int(cls): i for i, cls in enumerate(metrics.box.ap_class_index)}
print(f"{'Class':<23} {'P':>8} {'R':>8} {'AP50':>8} {'AP50:95':>10}")
for cls, name in enumerate(NAMES):
    support = DATASET_SUMMARY["image_counts"]["val"].get(name, 0)
    if support == 0 or cls not in metric_indices:
        print(f"{name:<23} N/A (no supported validation metric)")
        VALIDATION_REPORT["per_class"][name] = {"val_images": support, "metrics": None}
        continue
    precision, recall, ap50, ap = map(
        float, metrics.box.class_result(metric_indices[cls])
    )
    print(f"{name:<23} {precision:8.4f} {recall:8.4f} {ap50:8.4f} {ap:10.4f}")
    VALIDATION_REPORT["per_class"][name] = {
        "val_images": support,
        "precision": precision,
        "recall": recall,
        "ap50": ap50,
        "ap50_95": ap,
    }
(RUN_DIR / "validation_metrics.json").write_text(
    json.dumps(VALIDATION_REPORT, indent=2), encoding="utf-8"
)
print(
    "Precision/recall use the library's selected operating point, not a calibrated deployment threshold."
)
print("These are validation-selection metrics, NOT final test performance.")


In [ ]:
# Save the requested convenient Drive copy, preserving any earlier best.pt first.
# The run-specific best.pt and experiment metadata are already on Drive.
import os

DESTINATION = DRIVE_ROOT / "best.pt"
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f"No best checkpoint to save: {BEST_WEIGHTS}")
if DESTINATION.exists():
    backup = DRIVE_ROOT / ("best_previous_" + uuid.uuid4().hex + ".pt")
    shutil.copy2(DESTINATION, backup)
    if sha256_file(backup) != sha256_file(DESTINATION):
        raise OSError(
            "Previous checkpoint backup failed verification; refusing overwrite."
        )
    print("Preserved previous best.pt:", backup)
temporary = DRIVE_ROOT / ("best_" + uuid.uuid4().hex + ".tmp")
shutil.copy2(BEST_WEIGHTS, temporary)
expected_hash = sha256_file(BEST_WEIGHTS)
if sha256_file(temporary) != expected_hash:
    raise OSError("Checkpoint copy failed checksum verification.")
os.replace(temporary, DESTINATION)
if sha256_file(DESTINATION) != expected_hash:
    raise OSError("Final Drive checkpoint checksum mismatch.")
RUN_METADATA["best_weights_sha256"] = expected_hash
RUN_METADATA["best_weights_drive_path"] = str(DESTINATION)
RUN_METADATA["run_best_weights"] = str(BEST_WEIGHTS)
(RUN_DIR / "run_metadata.json").write_text(
    json.dumps(RUN_METADATA, indent=2), encoding="utf-8"
)
print(f"SUCCESS: best.pt saved to {DESTINATION}")
print(f"SHA-256: {expected_hash}")
print(f"Run-specific weights, settings, package versions, metrics and plots: {RUN_DIR}")


## Baseline v1 — dataset as of 2026-09-10

This is the **baseline v1 protocol** for the dataset as of **2026-09-10** (the currently documented preprocessing/synthetic build was produced on 2026-09-05). It becomes a trained baseline only after successful manual execution; use the printed manifest hash and recorded execution date to identify the actual uploaded snapshot.

**Crease, edge_damage and foreign_object have limited real validation/test coverage.** Crease and edge_damage each have only one independent real source per held-out split; tiles are correlated, not new observations. Foreign_object has more TILDA coverage than those two classes, but limited source diversity and especially sparse AITEX coverage. Consult `ai/datasets/README.md` for exact source limitations. Synthetic training images do not increase real held-out evidence.

This run includes synthetic train examples and weak rmshashi full-image boxes. It is not a strong-label-only localization benchmark. Pattern_break is a texture-anomaly proxy. Image-level splits do not establish unseen-roll generalization; reserved unmapped IDs have no measured AP. Respect source attribution and academic/non-commercial restrictions.

Validation selects checkpoints and early stopping, so these reported scores are **not an unbiased final test result**. Do not run test evaluation while tuning. A later controlled real-only versus real-plus-synthetic comparison and a frozen final-test protocol are still needed before claiming benefit or deployment accuracy.

Colab may disconnect; per-run best/last checkpoints and logs persist on Drive, but unsaved work within an epoch can be lost. Keep the notebook and run folder together, report failures honestly, and send the run metadata, validation JSON and curves to Zarwan's project handoff so `docs/Memory.md` can record actual results. No training or GPU verification was performed when this notebook was authored.